In [1]:
import os

os.environ['HF_ENDPOINT'] = 'http://hf-mirror.com'
# 下列两个任选一个
# os.environ['TRANSFORMERS_CACHE'] = '/opt/work/huggingface/hub'
# os.environ['XDG_CACHE_HOME'] = '/opt/work'
os.environ['XDG_CACHE_HOME'] = r'D:\huggingface'

In [2]:
from transformers import BertForSequenceClassification, BertTokenizer
from datasets import load_dataset
import torch
import torch.nn as nn
import numpy as np

D:\anaconda3\envs\gpu_default\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import transformers
transformers.__version__

'4.57.6'

## 模型迁移训练

### 加载原始csv格式数据

In [4]:
# 加载原始文本
dataset = load_dataset(
    "csv", # 给定格式
    data_dir="./datas/text_classify/intention",
    data_files={
        "train": "train.csv"
    },
    #error='xx'
    sep="\t",
    header=None,
    names=['text', 'label']
)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 12100
    })
})

In [5]:
print(type(dataset))
print(dataset['train'][0])

<class 'datasets.dataset_dict.DatasetDict'>
{'text': '还有双鸭山到淮阴的汽车票吗13号的', 'label': 'Travel-Query'}


### 构造标签映射mapping

In [6]:
# 遍历数据构造分类标签映射mapping
labels = set()
for item in dataset['train']:
    labels.add(item['label'])
labels = sorted(list(labels))
labelname2id = {label_name:label_idx for label_idx,label_name in enumerate(labels)}
print(labelname2id)

{'Alarm-Update': 0, 'Audio-Play': 1, 'Calendar-Query': 2, 'FilmTele-Play': 3, 'HomeAppliance-Control': 4, 'Music-Play': 5, 'Other': 6, 'Radio-Listen': 7, 'TVProgram-Play': 8, 'Travel-Query': 9, 'Video-Play': 10, 'Weather-Query': 11}


### 模型迁移-恢复分词器及模型

In [7]:
bert_path = r"D:\huggingface\huggingface\hub\models--bert-base-chinese"
# Bert模型迁移
tokenizer = BertTokenizer.from_pretrained(bert_path)
# 迁移模型，并且更改最终输出的类别数目
model = BertForSequenceClassification.from_pretrained(
    bert_path,
    num_labels=len(labelname2id), # 重新更改标签数目
    id2label=dict(enumerate(labels)), # 重新更改id和标签的映射mapping
    label2id=labelname2id, # 重新更改id和标签的映射mapping
    weights_only=False  # 这个参数weights_only在不同的transformers版本中不一定需要
)
print(model)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at D:\huggingface\huggingface\hub\models--bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

### 数据处理-分词+token id转换+标签转换

In [8]:
# 对数据进行分词转换
def preprocess_function(examples):
    """
    对单个文本进行分词转换
    """
    text, label = examples['text'], examples['label']
    # 分词转换
    item = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding=False
    )
    
    item["labels"] = torch.tensor(labelname2id[label])
    return item


# 应用预处理函数（num_proc=4表示多进程加速）
tokenized_dataset = dataset.map(
    preprocess_function,
    num_proc=None,
    remove_columns=['text', 'label'] # 删除列
)
print(tokenized_dataset)
print(tokenized_dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 12100
    })
})
{'input_ids': [101, 6820, 3300, 1352, 7890, 2255, 1168, 3917, 7346, 4638, 3749, 6756, 4873, 1408, 8124, 1384, 4638, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': 9}


In [9]:
# 数据抽样以及数据分割
#sample_dataset = tokenized_dataset['train'].shuffle(seed=24).take(n=1000)
sample_dataset = tokenized_dataset['train']

train_test_dataset = sample_dataset.train_test_split(
    test_size=0.2,
    seed=24
)
train_test_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 9680
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2420
    })
})

### 构造模型评估指标

In [10]:
# 定义评估指标
import evaluate # pip install evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    # 解析预测结果和标签 (前行输出以及实际标签值)
    predictions, labels = eval_pred
    # 将 logits 转换为预测标签（取最大值索引）
    predictions = np.argmax(predictions, axis=-1)

    # 计算指标
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")  # 多分类用 weighted

    # 返回指标字典（键为指标名称，值为数值）
    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"],
    }

ModuleNotFoundError: No module named 'evaluate'

In [11]:
def compute_metrics(eval_pred):
    # PS: 默认情况下，会自动将多个批次的数据合并成一个tensor/ndarray对象
    label_ids = eval_pred.label_ids  # 评估数据的实际标签值 [bs]
    predictions = eval_pred.predictions  # 评估数据经过模型前向的输出值 [bs,class_num]
    pred_ids = np.argmax(predictions, axis=-1)  # [bs,class_num] -> [bs]

    from sklearn import metrics
    f1 = metrics.f1_score(label_ids, pred_ids, average='micro')
    acc = metrics.accuracy_score(label_ids, pred_ids)
    return {
        "f1": f1,
        "acc": acc
    }


### 构造训练参数

In [12]:
from transformers import TrainingArguments, Trainer

# 可能需要安装：pip install transformers[torch]

training_args = TrainingArguments(
    output_dir="./output/bert-finetuned-intent-textclassify/models",  # 模型保存路径
    overwrite_output_dir=True,
    num_train_epochs=3,  # 训练轮数
    per_device_train_batch_size=4,  # 单设备训练批次大小（视GPU内存调整）
    per_device_eval_batch_size=4,   # 单设备验证批次大小
    gradient_accumulation_steps=4,  # 梯度累积（显存不足时增大，等效于增大batch_size）
    eval_strategy="epoch",    # 每轮结束后验证
    save_strategy="epoch",          # 每轮结束后保存模型
    logging_dir="./output/bert-finetuned-intent-textclassify/logs",           # 日志路径
    logging_steps=10,
    learning_rate=5e-5,             # 学习率（GPT类模型通常用2e-5 ~ 5e-5）
    weight_decay=0.01,              # 权重衰减（正则化）
    fp16=True,                      # 启用混合精度训练（需GPU支持）
    load_best_model_at_end=True,    # 训练结束后加载最佳模型
    metric_for_best_model="f1",  # 以准确率为判断标准 默认为损失
    greater_is_better=True,  # metric_for_best_model越高越好（损失则设为 False， 默认为损失）
)

### 迭代训练

In [13]:
from transformers import DataCollatorForTokenClassification, DataCollatorWithPadding

# 数据填充对象--> 将多条样本数据组合成一个批次
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 初始化Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,  # 传入数据整理器
    train_dataset=train_test_dataset["train"],
    eval_dataset=train_test_dataset["test"],
    compute_metrics=compute_metrics,  # 评估指标
)

# 开始训练
trainer.train()

D:\anaconda3\envs\gpu_default\Lib\site-packages\transformers\models\bert\modeling_bert.py:413: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Epoch,Training Loss,Validation Loss,F1,Acc
1,0.360200,0.238023,0.930165,0.930165
2,0.149600,0.234296,0.938017,0.938017
3,0.072500,0.244967,0.941736,0.941736


TrainOutput(global_step=1815, training_loss=0.21691504206210785, metrics={'train_runtime': 377.6663, 'train_samples_per_second': 76.893, 'train_steps_per_second': 4.806, 'total_flos': 322728516623520.0, 'train_loss': 0.21691504206210785, 'epoch': 3.0})

In [14]:
# 模型保存(最终的最优模型保存)
trainer.save_model(os.path.join(training_args.output_dir, "./final-best-model"))

## 模型推理应用

### 基于分词、模型的分步骤推理

In [15]:
local_path = "./output/bert-finetuned-intent-textclassify/models/final-best-model"
local_tokenizer = BertTokenizer.from_pretrained(local_path)
local_model = BertForSequenceClassification.from_pretrained(local_path)
# 数据填充对象
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
local_model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [16]:
# 针对给定文本进行分词预处理
text = "15号上午10点带孩子去海洋馆的行程帮我制定下。"
text = "还有双鸭山到淮阴的汽车票吗13号的"
item = local_tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding=False
    )
item

  

{'input_ids': [101, 6820, 3300, 1352, 7890, 2255, 1168, 3917, 7346, 4638, 3749, 6756, 4873, 1408, 8124, 1384, 4638, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [17]:
# 数据转换tensor
batch = data_collator([item])
batch

{'input_ids': tensor([[ 101, 6820, 3300, 1352, 7890, 2255, 1168, 3917, 7346, 4638, 3749, 6756,
         4873, 1408, 8124, 1384, 4638,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [18]:
# 调用模型
output = local_model(**batch)
logits = output.logits
print(logits)

pred_idx = torch.argmax( logits, dim=-1)[0].item() # 预测类别
pred_name = labels[pred_idx]

print(f"预测结果:{pred_idx}  {pred_name}")

tensor([[-0.5451, -0.4850, -0.8001, -1.9424, -0.2125, -1.3043, -0.2289, -0.7473,
         -0.6630,  9.1195, -1.9321, -0.4230]], grad_fn=<AddmmBackward0>)
预测结果:9  Travel-Query


### 基于pipeline的推理 一

In [19]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer

local_path = "./output/bert-finetuned-intent-textclassify/models/final-best-model"

# 恢复模型和分词器
model = AutoModelForSequenceClassification.from_pretrained(local_path)
tokenizer = AutoTokenizer.from_pretrained(local_path)

# 初始化 pipeline，指定模型和分词器
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    #return_all_scores=True  # 返回所有类别的得分（默认只返回最高分类别）
)
classifier

Device set to use cuda:0


In [20]:
classifier([
    "15号上午10点带孩子去海洋馆的行程帮我制定下。",
    "还有双鸭山到淮阴的汽车票吗13号的",
    "播放钢琴曲命运交响曲",
    "每天09:00关闭空调"
], top_k=2)

[[{'label': 'Alarm-Update', 'score': 0.9994184970855713},
  {'label': 'Travel-Query', 'score': 8.315208833664656e-05}],
 [{'label': 'Travel-Query', 'score': 0.9994032382965088},
  {'label': 'HomeAppliance-Control', 'score': 8.849154983181506e-05}],
 [{'label': 'Music-Play', 'score': 0.9969384670257568},
  {'label': 'Other', 'score': 0.0016024108044803143}],
 [{'label': 'HomeAppliance-Control', 'score': 0.9983017444610596},
  {'label': 'Alarm-Update', 'score': 0.0008740871562622488}]]

### 基于pipeline的推理 二

In [25]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
from pathlib import Path

local_path = "./output/bert-finetuned-intent-textclassify/models/final-best-model"

# 恢复模型和分词器
#model = AutoModelForSequenceClassification.from_pretrained(local_path)
#tokenizer = AutoTokenizer.from_pretrained(local_path)

# 初始化 pipeline，指定模型和分词器
classifier = pipeline(
    "text-classification",
    model=local_path,  # 本地模型路径
    #tokenizer=local_path  # 分词器路径（与模型同目录时可省略）
)
print(classifier)
print(classifier.model)

Device set to use cuda:0


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [26]:
classifier([
    "15号上午10点带孩子去海洋馆的行程帮我制定下。",
    "还有双鸭山到淮阴的汽车票吗13号的",
    "播放钢琴曲命运交响曲",
    "每天09:00关闭空调"
], top_k=2)

[[{'label': 'Alarm-Update', 'score': 0.9994184970855713},
  {'label': 'Travel-Query', 'score': 8.315208833664656e-05}],
 [{'label': 'Travel-Query', 'score': 0.9994032382965088},
  {'label': 'HomeAppliance-Control', 'score': 8.849154983181506e-05}],
 [{'label': 'Music-Play', 'score': 0.9969384670257568},
  {'label': 'Other', 'score': 0.0016024108044803143}],
 [{'label': 'HomeAppliance-Control', 'score': 0.9983017444610596},
  {'label': 'Alarm-Update', 'score': 0.0008740871562622488}]]

#### 遍历所有数据进行预测处理

In [27]:
# 加载原始文本
dataset = load_dataset(
    "csv",
    data_dir="./datas/text_classify/intention",
    data_files={
        "train": "train.csv"
    },
    #error='xx'
    sep="\t",
    header=None,
    names=['text', 'label']
)
dataset = dataset['train']

In [28]:
def batch_predict_map(examples):
    """
    对样本进行处理
    :param examples: Dict[str, Any] 或者 Dict[str, List[Any]]
    :return:
    """
    # 调用模型进行预测
    preds = classifier(examples['text'])
    pred_labels = [pred['label'] for pred in preds] # 多样样本的预测结果合并成list
    # 添加结果
    examples['pred_label'] = pred_labels

    return examples

dataset = dataset.map(
    batch_predict_map,
    batched=True,
    batch_size=10
)

Map: 100%|██████████| 12100/12100 [01:27<00:00, 138.85 examples/s]


In [29]:
dataset[0]

{'text': '还有双鸭山到淮阴的汽车票吗13号的',
 'label': 'Travel-Query',
 'pred_label': 'Travel-Query'}

In [30]:
# 获取预测失败的样本
def predict_error_filter(example):
    return example['label'] != example['pred_label']

dataset = dataset.filter(
    predict_error_filter
)

Filter: 100%|██████████| 12100/12100 [00:00<00:00, 191536.63 examples/s]


In [31]:
# 排序一下
dataset = dataset.sort(column_names=['label', 'pred_label'])

In [33]:
dataset.to_csv("./datas/text_classify/intention/error.csv", header=True, index=False)

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 500.27ba/s]


15176